## Update a drilling campaign object with interim data

This example combines the drilling campaign selection flow from the download sample with the interim publish flow from the create sample. It downloads an existing drilling campaign object, copies its planned section, builds a new interim section from CSV data, and publishes a new version of the object back to Evo.

> **Note:** This notebook requires an existing drilling campaign object in your Evo workspace with a populated `planned` section.
>
> The notebook uses `sample-data/interim.csv` to build the new interim section.

### Requirements

You must have a Seequent account with the Evo entitlement to use this notebook.

The following parameters must be provided:

- The client ID of your Evo application.
- The callback/redirect URL of your Evo application.

To obtain these app credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens) in the Seequent Developer Portal.

In [ ]:
import pandas as pd
from evo.notebooks import ServiceManagerWidget

input_path = "sample-data"
object_hole_id = "Holeid"

# Evo app credentials
client_id = "<your-client-id>"
redirect_url = "<your-redirect-url>"

manager = await ServiceManagerWidget.with_auth_code(
    redirect_url=redirect_url,
    client_id=client_id,
).login()

In [ ]:
import copy
import uuid
from datetime import datetime
from uuid import UUID

import ipywidgets as widgets
import numpy as np
from IPython.display import JSON, display

from evo.notebooks import FeedbackWidget
from evo.objects import ObjectAPIClient

object_client = ObjectAPIClient(manager.get_environment(), manager.get_connector())
data_client = object_client.get_data_client(manager.cache)


def create_hole_id_mapping(hole_id_table, value_list):
    num_keys = len(hole_id_table.index)

    mapping_df = pd.DataFrame(list())
    mapping_df["hole_index"] = hole_id_table["key"]
    mapping_df["offset"] = [0] * num_keys
    mapping_df["count"] = [0] * num_keys

    mapping_df["hole_index"] = mapping_df["hole_index"].astype("int32")
    mapping_df["offset"] = mapping_df["offset"].astype("uint64")
    mapping_df["count"] = mapping_df["count"].astype("uint64")

    prev_value = ""
    key = ""
    count = 0
    offset = 0

    for _, row in value_list.iterrows():
        new_value = row["data"]

        if new_value != prev_value:
            if prev_value != "":
                mapping_df.loc[mapping_df["hole_index"] == key, "count"] = count
                mapping_df.loc[mapping_df["hole_index"] == key, "offset"] = offset
                offset += count

            mask = hole_id_table["value"] == new_value
            masked_df = hole_id_table[mask]
            try:
                key_row = masked_df.iloc[[0]]
            except IndexError:
                print("Ignoring this hole ID")
                continue

            key = key_row["key"].iloc[0]
            count = 1
            prev_value = new_value
        else:
            count += 1

    mapping_df.loc[mapping_df["hole_index"] == key, "count"] = count
    mapping_df.loc[mapping_df["hole_index"] == key, "offset"] = offset

    return mapping_df


def create_category_lookup_and_values(attribute):
    attribute = attribute.copy()
    attribute.replace(np.nan, "", regex=True, inplace=True)
    list_obj = sorted(set(attribute["data"]))

    table_df = pd.DataFrame([])
    table_df["key"] = list(range(0, len(list_obj)))
    table_df["value"] = list_obj

    values_df = pd.DataFrame([])
    values_df["data"] = attribute["data"].map(table_df.set_index("value")["key"])
    return table_df, values_df


def serialize_for_display(obj):
    if isinstance(obj, UUID):
        return str(obj)
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, list):
        return [serialize_for_display(item) for item in obj]
    if isinstance(obj, dict):
        return {key: serialize_for_display(value) for key, value in obj.items()}
    if hasattr(obj, "__dict__"):
        return {key: serialize_for_display(value) for key, value in vars(obj).items()}
    return obj


async def download_table(table_info, label):
    return (
        await data_client.download_table(
            object_id=metadata.id,
            version_id=None,
            table_info=table_info,
            fb=FeedbackWidget(label),
        )
    ).to_pandas()

### Load all drilling campaigns in the workspace

The next cell lists drilling campaign objects in the currently selected Evo workspace and displays a dropdown so you can choose one to update.

In [ ]:
all_objects = await object_client.list_all_objects()
drilling_campaigns = [obj for obj in all_objects if "drilling-campaign" in obj.schema_id.sub_classification]

if len(drilling_campaigns) == 0:
    print("No drilling campaigns found in the selected workspace.")
else:
    dropdown_options = [(obj.name.removesuffix(".json"), str(obj.id)) for obj in drilling_campaigns]

    drilling_campaign_dropdown = widgets.Dropdown(
        options=dropdown_options,
        description="Drilling campaign:",
        disabled=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    )

    print(f"Found {len(drilling_campaigns)} drilling campaign(s). Please select one:")
    display(drilling_campaign_dropdown)

### Download the selected drilling campaign JSON

Select a drilling campaign in the dropdown above, then run the next cell to download the existing object JSON.

In [ ]:
selected_object_id = drilling_campaign_dropdown.value
if selected_object_id is None:
    raise ValueError("Select a drilling campaign before running this cell.")

downloaded_object = await object_client.download_object_by_id(object_id=selected_object_id, version=None)
metadata = downloaded_object.metadata
downloaded_dict = downloaded_object.as_dict()

display(JSON(serialize_for_display(downloaded_dict), expanded=False))
print(f"Selected object: {downloaded_dict['name']}")
print(f"Object ID: {metadata.id}")
print(f"Current version: {metadata.version_id}")

In [ ]:
from evo_schemas.components import (
    CategoryAttribute_V1_1_0,
    DownholeDirectionVector_V1_0_0,
    HoleChunks_V1_0_0,
    HoleCollars_V1_0_0,
    NanCategorical_V1_0_1,
    StringAttribute_V1_1_0,
)
from evo_schemas.elements import FloatArray3_V1_0_1, IntegerArray1_V1_0_1, LookupTable_V1_0_1, StringArray_V1_0_1
from evo_schemas.objects import DrillingCampaign_V1_0_0_Interim

if "planned" not in downloaded_dict:
    raise ValueError("The selected drilling campaign does not contain a planned section.")

planned_section = copy.deepcopy(downloaded_dict["planned"])
planned_collar = planned_section["collar"]
hole_id_table_df = await download_table(downloaded_dict["hole_id"]["table"], "Downloading hole ID lookup table")

coordinates_component = FloatArray3_V1_0_1.from_dict(planned_collar["coordinates"])
distances_component = FloatArray3_V1_0_1.from_dict(planned_collar["distances"])

interim_file_path = f"{input_path}/interim.csv"
interim_df = pd.read_csv(interim_file_path)
interim_df = interim_df.sort_values([object_hole_id]).reset_index(drop=True)
interim_df = interim_df[interim_df[object_hole_id].isin(hole_id_table_df["value"])].reset_index(drop=True)

if interim_df.empty:
    raise ValueError("No interim rows remain after matching Hole IDs to the selected drilling campaign.")

survey_attributes = [
    {
        "name": "Wolfpass GM",
        "key": str(uuid.uuid4()),
        "type": "category",
    }
]

survey_hole_values_df = pd.DataFrame({"data": interim_df[object_hole_id]})
interim_hole_chunks_df = create_hole_id_mapping(hole_id_table=hole_id_table_df, value_list=survey_hole_values_df)
interim_hole_chunks_component = HoleChunks_V1_0_0.from_dict(data_client.save_dataframe(interim_hole_chunks_df))

survey_distances_df = interim_df[["depth", "azimuth", "dip"]].copy()
survey_distances_df = survey_distances_df.rename(columns={"depth": "distance"})
survey_distances_df = survey_distances_df.astype(float).reset_index(drop=True)
interim_survey_component = DownholeDirectionVector_V1_0_0.from_dict(data_client.save_dataframe(survey_distances_df))

interim_attributes = []
for survey_attribute in survey_attributes:
    attribute_name = survey_attribute["name"]
    attribute_key = survey_attribute["key"]
    attribute_type = survey_attribute["type"]

    attribute_df = pd.DataFrame({"data": interim_df[attribute_name]})

    if attribute_type == "string":
        attribute_df = attribute_df.astype(str).reset_index(drop=True)
        values = StringArray_V1_0_1.from_dict(data_client.save_dataframe(attribute_df))
        attribute = StringAttribute_V1_1_0(name=attribute_name, key=attribute_key, values=values)
    elif attribute_type == "category":
        attribute_df = attribute_df.astype(str).reset_index(drop=True)
        table_df, values_df = create_category_lookup_and_values(attribute_df)

        table = LookupTable_V1_0_1.from_dict(data_client.save_dataframe(table_df))
        values = IntegerArray1_V1_0_1.from_dict(data_client.save_dataframe(values_df))
        attribute = CategoryAttribute_V1_1_0(
            name=attribute_name,
            key=attribute_key,
            nan_description=NanCategorical_V1_0_1(values=[]),
            table=table,
            values=values,
        )
    else:
        raise ValueError(f"Unsupported attribute type: {attribute_type}")

    interim_attributes.append(attribute)

interim_hole_collars_component = HoleCollars_V1_0_0(
    coordinates=coordinates_component,
    distances=distances_component,
    holes=interim_hole_chunks_component,
    attributes=interim_attributes,
)

interim_component = DrillingCampaign_V1_0_0_Interim(
    collar=interim_hole_collars_component,
    path=interim_survey_component,
)

interim_component.model_dump(mode="python", by_alias=True, exclude_none=True)

In [ ]:
updated_object_dict = copy.deepcopy(downloaded_dict)
updated_object_dict["planned"] = planned_section
updated_object_dict["interim"] = interim_component.model_dump(mode="python", by_alias=True, exclude_none=True)

await data_client.upload_referenced_data(updated_object_dict, FeedbackWidget("Uploading interim data"))
updated_object = await downloaded_object.update(updated_object_dict)

print(f"Successfully published a new version of '{updated_object_dict['name']}'.")
print(f"New version ID: {updated_object.metadata.version_id}")

## Summary

In this example, you:

- Logged in and selected an Evo instance, workspace, and drilling campaign object.
- Downloaded the existing drilling campaign JSON.
- Copied the planned section from the existing object.
- Built a new interim section from CSV data.
- Published a new version of the drilling campaign back to the same workspace.